# OceanSODA (NOAA) Dataset processing. 
The SODA data is used for the DIC and Alk restoring forces. This PR reduces the resolution, time, and number of variables in order to reduce size of the test data file. 

In [1]:
import xarray as xr
import fsspec
import numpy as np

url = "https://www.ncei.noaa.gov/data/oceans/archive/arc0160/0220059/6.6/data/0-data/OceanSODA_ETHZ-v2025.OCADS.01-1982-2024.nc"
ds = xr.open_dataset(fsspec.open(url).open(), engine="h5netcdf")

In [2]:
# Reduce to 2 deg resolution
ds = ds.isel(lat=slice(None, None, 3), lon=slice(None, None, 3))

In [3]:
variables = list(ds.variables) 
keep = {'talk','dic','temperature','salinity', 'year', 'lat', 'lon', 'time'}
drop_list = [item for item in variables if item not in keep]
drop_list

['spco2',
 'sfco2',
 'ph_total',
 'ph_free',
 'hco3',
 'co3',
 'co2',
 'revelle_factor',
 'omega_ca',
 'omega_ar',
 'dic_uncert',
 'spco2_uncert',
 'ph_total_uncert',
 'omega_ca_uncert',
 'omega_ar_uncert',
 'talk_uncert',
 'sfco2_uncert']

In [4]:
ds = ds.drop_vars(drop_list)

In [5]:
ds

<xarray.Dataset> Size: 59MB
Dimensions:      (time: 516, lat: 60, lon: 120)
Coordinates:
  * lat          (lat) float64 480B -89.5 -86.5 -83.5 -80.5 ... 81.5 84.5 87.5
  * lon          (lon) float64 960B -179.5 -176.5 -173.5 ... 171.5 174.5 177.5
  * time         (time) datetime64[ns] 4kB 1982-01-15 1982-02-15 ... 2024-12-15
    year         (time) int64 4kB ...
Data variables:
    talk         (time, lat, lon) float32 15MB ...
    dic          (time, lat, lon) float32 15MB ...
    temperature  (time, lat, lon) float32 15MB ...
    salinity     (time, lat, lon) float32 15MB ...
Attributes:
    contact:      gregorl@ethz.ch
    author:       Luke Gregor
    institution:  ETH Zuerich
    version:      v2025.OCADS
    date:         2025-10-09
    description:  talk and pco2 (more accurately fco2) are estimated with two...
    changelog:    v2021d: Extended from 1982-2020; Now using: OISSTv2.1 for S...
    reference:    Gregor, L. and Gruber, N.: OceanSODA-ETHZ: A global gridded...
    source:       https://doi.org/10.25921/m5wx-ja34
    product:      OSETHZ-v2025.OCADS

In [7]:
ds = ds.load()

In [8]:
ds.to_netcdf('./updated_files/coarsened_OceanSODA_dataset.nc')